# E1 - Softmax, MLP, CNN

Fashion-MNIST classification; explicit training loop; four model variants; quantitative and qualitative error analysis.

> **Default:** notebooks run in `SMOKE=True` mode so a fresh clone can validate structure without downloading large datasets. Switch to `SMOKE=False` for real experiments.

## Session 0 - Environment and reproducibility

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SMOKE = True
SEED = 42
print("repo:", ROOT)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
from common.seed import seed_everything, resolve_device
seed_everything(SEED); device=resolve_device(); device

## Session 1 - Dataset, split and EDA
Full mode downloads Fashion-MNIST. Smoke mode uses synthetic tensors only.

In [ ]:
from torch.utils.data import DataLoader
from common.synthetic import classification_dataset
if SMOKE:
    ds=classification_dataset(n=48); train_loader=DataLoader(ds,batch_size=16,shuffle=True); val_loader=DataLoader(ds,batch_size=16); test_loader=val_loader
else:
    from E1.src.data import fashion_mnist_loaders
    train_loader,val_loader,test_loader=fashion_mnist_loaders(root=ROOT/"data",batch_size=128)
len(train_loader), len(val_loader), len(test_loader)

## Session 2 - Model definitions and shape sanity checks

In [ ]:
from E1.src.models import build_e1_models
models=build_e1_models()
x,y=next(iter(train_loader))
{name: tuple(model(x[:2]).shape) for name,model in models.items()}

## Session 3 - Training protocol

In [ ]:
from torch import nn
from common.engine import fit_classifier, evaluate_classifier
name="cnn_2block"
model=models[name].to(device)
criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
history=fit_classifier(model,train_loader,val_loader,optimizer,criterion,device,epochs=1 if SMOKE else 20)
history[-1]

## Session 4 - Test evaluation and artifacts

In [ ]:
metrics, y_true, y_pred=evaluate_classifier(model,test_loader,criterion,device)
print(metrics)
# Full mode: export history CSV, learning curves, confusion matrix and misclassified-image gallery to E1/outputs.

## Session 5 - Report checklist
- Accuracy + number of parameters/capacity
- Loss/accuracy curves
- Confusion matrix / wrong samples
- Short analysis, not accuracy-only